# MyDigitalTwin — Analyse Fréquentielle des Centres d'Intérêt

**Objectif** : Identifier les concepts les plus fréquents dans mes données réelles (Spotify, YouTube, Google, Netflix, Chrome) pour calibrer les macro-catégories de la home page.

**Approche** : Analyse de fréquence de mots par source → identification des gaps dans `CATEGORY_KEYWORDS` → mise à jour du dictionnaire.

> **Pourquoi pas K-Means ?**  
> Une première tentative de clustering TF-IDF + K-Means a produit un cluster *catch-all* dominant (~73% des données, Silhouette ≈ 0.19). Le problème est structurel : les textes courts multi-sources (artistes Spotify, titres YouTube, requêtes Google) ont des espaces sémantiques trop hétérogènes pour être clusterisés conjointement.  
> L'analyse fréquentielle est plus directe et interprétable pour des catégories prédéfinies.

In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MyDigitalTwin-FrequencyAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(table_name):
    return spark.read.format("delta").load(os.path.join(WAREHOUSE, table_name))

---
## Étape 1 — Chargement des sources texte

On extrait la colonne texte pertinente de chaque source Delta, en gardant la provenance (`source`) pour analyser chaque canal séparément.

In [ ]:
# ── 1. CHARGEMENT ─────────────────────────────────────────────────────────────
sources = {
    "Google Searches": read_table("google_searches").select(F.col("query").alias("text")),
    "YouTube":         read_table("youtube_watch").select(F.col("title").alias("text")),
    "Chrome":          read_table("google_chrome").select(F.col("title").alias("text")),
    "Spotify":         read_table("spotify_streams").select(F.col("artistName").alias("text")).dropDuplicates(["text"]),
    "Netflix":         read_table("netflix_views").select(F.col("show_title").alias("text")),
}

for name, df in sources.items():
    print(f"{name:20s}: {df.count():>6,} lignes")

In [ ]:
# ── 2. ANALYSE FRÉQUENTIELLE PAR SOURCE ───────────────────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Stopwords : anglais + français + bruit technique
STOPWORDS_EXTRA = [
    # Français
    "les", "des", "une", "sur", "avec", "dans", "qui", "que", "par", "plus",
    "tout", "bien", "comme", "mais", "mon", "ton", "son", "nos", "mes",
    "faire", "comment", "plus", "aussi", "encore", "très",
    # Anglais générique
    "the", "and", "for", "with", "you", "your", "this", "that", "from",
    "are", "was", "not", "its", "but", "all", "new", "best", "how",
    # Bruit technique (URLs, tracking, ads)
    "https", "http", "www", "com", "org", "net", "html", "php", "utm",
    "amp", "utm_source", "befr", "dgoogle", "watch", "video", "clip",
    "official", "officiel", "youtube", "shorts",
]

STOP_ALL = StopWordsRemover.loadDefaultStopWords("english") \
         + StopWordsRemover.loadDefaultStopWords("french") \
         + STOPWORDS_EXTRA

results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))   # exclure les URLs brutes
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    freq = (
        filtered
        .select(F.explode("tokens").alias("word"))
        .filter(F.length("word") > 2)
        .groupBy("word")
        .count()
        .orderBy(F.desc("count"))
    )

    results[source_name] = freq
    print(f"✓ {source_name}")

print("\nAnalyse terminée.")

In [ ]:
# ── 3. TOP 25 MOTS PAR SOURCE ─────────────────────────────────────────────────
TOP_N = 25

for source_name, freq_df in results.items():
    print(f"\n{'='*50}")
    print(f"  {source_name}")
    print(f"{'='*50}")
    freq_df.show(TOP_N, truncate=False)

In [ ]:
# ── 4. BIGRAMMES — Termes composés importants ─────────────────────────────────
# Les bigrammes capturent des concepts que les mots seuls manquent :
# "travis scott", "league of legends", "formula 1", "deep learning", etc.
from pyspark.ml.feature import NGram

bigram_results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    bigrams = NGram(n=2, inputCol="tokens", outputCol="ngrams").transform(filtered)

    freq = (
        bigrams
        .select(F.explode("ngrams").alias("bigram"))
        .filter(F.length("bigram") > 5)
        .groupBy("bigram")
        .count()
        .orderBy(F.desc("count"))
    )

    bigram_results[source_name] = freq

print("Top bigrammes par source :\n")
for source_name, freq_df in bigram_results.items():
    print(f"── {source_name}")
    freq_df.show(15, truncate=False)
    print()

In [ ]:
# ── 5. GAP ANALYSIS — Termes fréquents non couverts par CATEGORY_KEYWORDS ─────
# CATEGORY_KEYWORDS est défini dans config.yaml — modifier là-bas pour personnaliser.
from config import CATEGORY_KEYWORDS

all_kw = {kw for kws in CATEGORY_KEYWORDS.values() for kw in kws}

# Union de toutes les sources pour la vue globale
from functools import reduce
all_sources = reduce(lambda a, b: a.union(b), sources.values())
clean_all = all_sources.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 2) &
    (~F.col("text").rlike(r'^https?://'))
)
tokenized_all = Tokenizer(inputCol="text", outputCol="words").transform(clean_all)
filtered_all  = StopWordsRemover(inputCol="words", outputCol="tokens", stopWords=STOP_ALL).transform(tokenized_all)

global_freq = (
    filtered_all
    .select(F.explode("tokens").alias("word"))
    .filter(F.length("word") > 2)
    .groupBy("word").count()
    .orderBy(F.desc("count"))
)

top_words = [row["word"] for row in global_freq.limit(200).collect()]
uncovered = [w for w in top_words if w not in all_kw]

print("Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :")
print("(candidats à ajouter dans une catégorie)
")
for w in uncovered[:40]:
    print(f"  {w}")

In [1]:
spark.stop()
print("Spark session fermée.")

Spark session fermée.
